# C8-embeddings — Practice p10 — Solution

In [ ]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["bread", "cheese", "soup", "honey", "pepper", "garlic",
         "hammer", "nail", "saw", "drill"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
S = W @ W.T
sym_gap = float(np.max(np.abs(S - S.T)))
diag_gap = float(np.max(np.abs(np.diag(S) - 1.0)))

off_diagonal = np.where(np.eye(len(WORDS), dtype=bool), -np.inf, S)
ci, cj = divmod(int(np.argmax(off_diagonal)), len(WORDS))
closest_pair = (WORDS[ci], WORDS[cj])
closest_sim = float(S[ci, cj])

# Unit diagonal entries are row maxima, so none can be the minimum.
fi, fj = divmod(int(np.argmin(S)), len(WORDS))
farthest_pair = (WORDS[fi], WORDS[fj])
farthest_sim = float(S[fi, fj])

The matrix is symmetric to the computed precision and has a unit diagonal. Row-major flattening encounters the upper-triangle copy of each symmetric extreme first, so both reported pairs have the required increasing index order.

### Answer check

In [ ]:
assert W.shape == (10, 100) and S.shape == (10, 10)
assert sym_gap <= 1e-12 and diag_gap <= 1e-12
assert closest_pair == ("pepper", "garlic")
assert np.isclose(closest_sim, 0.8210249196647396, atol=1e-12, rtol=0)
assert farthest_pair == ("garlic", "hammer")
assert np.isclose(farthest_sim, 0.041702593364820253, atol=1e-12, rtol=0)
assert ci < cj and fi < fj